I want to see if models can do credit assignment all by themselves through replay.

In [1]:
import mlx.core as mx
from mlx_lm import load
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

model, tokenizer = load("mlx-community/Qwen2.5-Coder-7B-Instruct-4bit")

def ocsd(messages):
    ids = mx.array(tokenizer.apply_chat_template(messages))
    hidden = model.model(mx.concat([ids, ids])[None])[0, len(ids):]
    logits = model.lm_head(hidden)
    probs = mx.softmax(logits, axis=-1)
    labels = ids[1:]
    pred_ids = mx.argsort(probs, axis=-1)[:, :-(3 + 1):-1]
    mx.eval(pred_ids)

    rows = []
    for i in range(len(labels)):
        def cell(id):
            token = tokenizer.decode([id.tolist()], skip_special_tokens=False)
            return f"{token!r} @ {probs[i, id]:.3f}"
        
        row = {"label": cell(labels[i])}
        for j, pred_id in enumerate(pred_ids[i]):
            row[f"pred_{j}"] = cell(pred_id)
        rows.append(row)
    df = pd.DataFrame(rows)
    return df

ocsd([
    {
        "role": "user",
        "content": "Pick two numbers."
    },
    {
        "role": "assistant",
        "content": "2 and 3."
    },
    {
        "role": "user",
        "content": "Incorrect! The first number must be odd and the second number must be even."
    }
])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

,label,pred_0,pred_1,pred_2
0,'system' @ 0.000,'<|im_start|>' @ 0.408,'<|im_end|>' @ 0.276,'1' @ 0.135
1,'\n' @ 0.966,'\n' @ 0.966,'<|im_end|>' @ 0.011,'<|im_start|>' @ 0.007
2,'You' @ 0.002,'I' @ 0.633,'My' @ 0.125,'1' @ 0.053
3,' are' @ 0.227,"""'re"" @ 0.769",' are' @ 0.227,' got' @ 0.001
4,' Q' @ 0.000,' correct' @ 0.816,' right' @ 0.168,' absolutely' @ 0.012
5,'wen' @ 1.000,'wen' @ 1.000,'W' @ 0.000,' Wen' @ 0.000
6,"',' @ 0.985","',' @ 0.985",'.' @ 0.006,'<|im_end|>' @ 0.002
7,' created' @ 0.395,' created' @ 0.395,' an' @ 0.354,' a' @ 0.090
8,' by' @ 1.000,' by' @ 1.000,' to' @ 0.000,' and' @ 0.000
9,' Alibaba' @ 1.000,' Alibaba' @ 1.000,' Ali' @ 0.000,'阿里' @ 0.000


There's credit attribution, but it falls off after the first instance because the model's disagreement isn't reflected in the trajectory and so it tries to conform with what's already there, even if it's incorrect. I think this means we can't do a naive offline approach. Maybe we can try chunks?

Let's try a successful trajectory with some unnecessary steps.

In [2]:
ocsd([
    {
        "role": "user",
        "content": "What is the capital of Foo?"
    },
    {
        "role": "assistant",
        "content": "Ethan is cool."
    },
    {
        "role": "user",
        "content": "Huh."
    },
    {
        "role": "assistant",
        "content": "I love tacos."
    },
    {
        "role": "user",
        "content": "Okay."
    },
    {
        "role": "assistant",
        "content": "The capital of Foo is Bar."
    },
    {
        "role": "user",
        "content": "Correct!"
    }
])

,label,pred_0,pred_1,pred_2
0,'system' @ 0.000,'<|im_start|>' @ 0.504,'<|im_end|>' @ 0.438,':\n' @ 0.022
1,'\n' @ 0.974,'\n' @ 0.974,'<|im_end|>' @ 0.012,'<|im_start|>' @ 0.005
2,'You' @ 0.016,'Thank' @ 0.467,'I' @ 0.326,'Yes' @ 0.048
3,' are' @ 0.192,"""'re"" @ 0.680",' are' @ 0.192,' got' @ 0.085
4,' Q' @ 0.007,' correct' @ 0.810,' right' @ 0.060,' welcome' @ 0.043
5,'wen' @ 1.000,'wen' @ 1.000,' Wen' @ 0.000,'W' @ 0.000
6,"',' @ 0.996","',' @ 0.996",'<|im_end|>' @ 0.002,'.' @ 0.001
7,' created' @ 0.378,' created' @ 0.378,' an' @ 0.309,' a' @ 0.272
8,' by' @ 1.000,' by' @ 1.000,'<|im_end|>' @ 0.000,' and' @ 0.000
9,' Alibaba' @ 1.000,' Alibaba' @ 1.000,' Ali' @ 0.000,'阿里' @ 0.000


There is, again, credit attribution: it knows the capital of Foo is Bar and it wants to answer immediately ("The" or "Bar"). But after the first unnecessary step, the model once again tries to conform to the trajectory. I also just noticed, the model sometimes tries to end the conversation before fulfulling the user's request. I wonder why that is.

Anyway, I want to try the chunking idea. I feel like a model would kind of struggle to "go back in time" given just chunk (as opposed to a full chat history), so I want to try messing with position IDs.

In [3]:
from mlx_lm.models.qwen2 import Attention, Qwen2Model, ModelArgs
from mlx_lm.models.base import scaled_dot_product_attention
from typing import Any, Optional
from mlx_lm import generate

class MyRoPE:
    def __init__(self, dims: int, base: float = 10000):
        self.base = base
        self.inv_freq = 1 / (base ** (mx.arange(0, dims, 2) / dims))

    def __call__(self, x, position_ids: mx.array, offset: int = 0):
        B, N, L, D = x.shape

        position_ids = position_ids[:, offset:offset + L]

        # Fast path
        offset = position_ids[0, 0].item()
        if mx.all(position_ids == mx.arange(offset, offset + L)).item():
            return mx.fast.rope(
                x,
                D,
                traditional=False,
                base=self.base,
                scale=1.0,
                offset=offset,
            )

        angles = position_ids[:, None, :, None] * self.inv_freq
        cos = mx.cos(angles).astype(x.dtype)
        sin = mx.sin(angles).astype(x.dtype)

        u, v = x[..., :D // 2], x[..., D // 2:]
        return mx.concat([u * cos - v * sin, v * cos + u * sin], axis=-1)

def my_attention_init(self, args: ModelArgs):
    self._base_init(args)
    self.my_rope = MyRoPE(
        dims=args.hidden_size // args.num_attention_heads,
        base=args.rope_theta,
    )

def my_attention_call(
    self,
    x: mx.array,
    mask: Optional[mx.array] = None,
    cache: Optional[Any] = None,
) -> mx.array:
    position_ids = getattr(self, "_position_ids", None)
    if position_ids is None:
        return self._base_call(x, mask, cache)

    B, L, D = x.shape

    queries, keys, values = self.q_proj(x), self.k_proj(x), self.v_proj(x)

    # Prepare the queries, keys and values for the attention computation
    queries = queries.reshape(B, L, self.n_heads, -1).transpose(0, 2, 1, 3)
    keys = keys.reshape(B, L, self.n_kv_heads, -1).transpose(0, 2, 1, 3)
    values = values.reshape(B, L, self.n_kv_heads, -1).transpose(0, 2, 1, 3)

    if cache is not None:
        queries = self.my_rope(queries, position_ids=position_ids, offset=cache.offset)
        keys = self.my_rope(keys, position_ids=position_ids, offset=cache.offset)
        keys, values = cache.update_and_fetch(keys, values)
    else:
        queries = self.my_rope(queries, position_ids=position_ids)
        keys = self.my_rope(keys, position_ids=position_ids)

    output = scaled_dot_product_attention(
        queries, keys, values, cache=cache, scale=self.scale, mask=mask
    )
    output = output.transpose(0, 2, 1, 3).reshape(B, L, -1)
    return self.o_proj(output)

if getattr(Attention, "_base_init", None) is None:
    Attention._base_init = Attention.__init__
Attention.__init__ = my_attention_init

if getattr(Attention, "_base_call", None) is None:
    Attention._base_call = Attention.__call__
Attention.__call__ = my_attention_call

def my_model_call(
    self,
    inputs: mx.array,
    cache=None,
    input_embeddings: Optional[mx.array] = None,
    position_ids: Optional[mx.array] = None,
):
    if position_ids is None:
        position_ids = getattr(self, "_position_ids", None)
    
    for layer in self.layers:
        layer.self_attn._position_ids = position_ids
    
    return Qwen2Model._base_call(self, inputs, cache, input_embeddings)

if getattr(Qwen2Model, "_base_call", None) is None:
    Qwen2Model._base_call = Qwen2Model.__call__
Qwen2Model.__call__ = my_model_call

patched_model, _ = load("mlx-community/Qwen2.5-Coder-7B-Instruct-4bit")

messages = [{"role": "user", "content": "Tell me about the capital of France."}]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(generate(model, tokenizer, prompt=prompt, max_tokens=20))

patched_model.model._position_ids = mx.arange(100)[None]
print(generate(patched_model, tokenizer, prompt=prompt, max_tokens=20))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

The capital of France is Paris. It is located in the northern central part of the country and is
The capital of France is Paris. It is located in the northern central part of the country and is


In [4]:
mx.clear_cache()

In [5]:
def chunked_ocsd(messages):
    start_ids = tokenizer.encode("<|im_start|>assistant\n")
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    ids = tokenizer.apply_chat_template(messages)
    for i in range(len(start_ids), len(ids)):
        if ids[i - len(start_ids):i] != start_ids:
            continue

        input_ids = ids + start_ids
        position_ids = (
            list(range(len(ids))) +
            list(range(i - len(start_ids), i))
        )
        while True:
            hidden = patched_model.model(
                mx.array(input_ids)[None],
                position_ids=mx.array(position_ids)[None],
            )[0, -1]
            logits = patched_model.lm_head(hidden)
            id = mx.argmax(logits).item()
            if id == im_end_id:
                break
            input_ids += [id]
            position_ids += [position_ids[-1] + 1]
        
        print(tokenizer.decode(input_ids, skip_special_tokens=False))

chunked_ocsd([
    {
        "role": "user",
        "content": "Pick two numbers."
    },
    {
        "role": "assistant",
        "content": "2 and 3."
    },
    {
        "role": "user",
        "content": "Incorrect! The first number must be odd and the second number must be even."
    }
])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Pick two numbers.<|im_end|>
<|im_start|>assistant
2 and 3.<|im_end|>
<|im_start|>user
Incorrect! The first number must be odd and the second number must be even.<|im_end|>
<|im_start|>assistant
I


I tried a bunch of variants of this, trying different ways to "time travel" the model back to the chunk and different sets of messages to see their effects, but I think the overarching issues are that the original and "time travel" tokens just combine into one larger, vaguer token mathematically and that the model is a causal LM, so the model isn't conditioned to do anything with the tokens after the time traveled to. I think also I've realized LMs are smart enough that we shouldn't be mucking around with the internals of them to get a desired effect. But still, how do we get a model to put itself in the shoes of itself some time before?

In [6]:
def prompt_ocsd(messages):
    base_prompt = tokenizer.apply_chat_template(
        [
            {**message, "content": f"MESSAGE {i + 1}\n{message["content"]}"}
            for i, message in enumerate(messages)
        ],
        tokenize=False,
    )

    for i in range(len(messages)):
        if messages[i]["role"] != "assistant":
            continue
        
        prompt = base_prompt + f"<|im_start|>assistant\nMESSAGE {i + 1}\n"
        print(generate(model, tokenizer, prompt, max_tokens=100))

prompt_ocsd([
    {
        "role": "user",
        "content": "Pick two numbers."
    },
    {
        "role": "assistant",
        "content": "2 and 3."
    },
    {
        "role": "user",
        "content": "Incorrect! The first number must be odd and the second number must be even."
    }
])

1 and 4.<|im_end|>



Wow, this is really promising...

In [7]:
prompt_ocsd([
    {
        "role": "user",
        "content": "What is the capital of Foo?"
    },
    {
        "role": "assistant",
        "content": "Ethan is cool."
    },
    {
        "role": "user",
        "content": "Huh."
    },
    {
        "role": "assistant",
        "content": "I love tacos."
    },
    {
        "role": "user",
        "content": "Okay."
    },
    {
        "role": "assistant",
        "content": "The capital of Foo is Bar."
    },
    {
        "role": "user",
        "content": "Correct!"
    }
])

Ethan is cool.<|im_end|>

I love tacos.<|im_end|>

Thank you! If you have any more questions or need further assistance, feel free to ask.<|im_end|>



I guess nothing's telling the model to do better between runs? Since the feedback is implicit? But at the same time, I'm not sure the model understands its own causality. I tried a couple variants of prompting and I feel like the model tends to conform much more to the earlier messages than the later ones, making me think the model behaves this way because it feels like the user's responses are fixed.

Anyway, I had a conversation today with my good friend [Victor](https://usaco.org/index.php?page=finalists24) and at some point during our conversation I thought I heard him say "mask" (he said "math"). And then I had a really good idea, which is that I could mask out the tokens that the model didn't agree with so that it wouldnt influence the model to conform to what was already there.

In [ ]:
from mlx_lm.models.cache import make_prompt_cache

def masked_ocsd(messages):
    start_ids = tokenizer.encode("<|im_start|>assistant\n")
    end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    ids = tokenizer.apply_chat_template(messages)
    cache = make_prompt_cache(model)
    logits = model(mx.array(ids)[None], cache=cache)
    output = []
    pred = False
    for i, id in enumerate(ids):
        if ids[i - len(start_ids):i] == start_ids:
            pred = True

        next_id = mx.argmax(logits[0, -1], axis=-1).item() if pred else id
        logits = model(mx.array([next_id])[None], cache=cache)
        output.append(next_id)

        if id == end_id:
            pred = False

    print(tokenizer.decode(output))

masked_ocsd([
    {
        "role": "user",
        "content": "Pick two numbers."
    },
    {
        "role": "assistant",
        "content": "2 and 3."
    },
    {
        "role": "user",
        "content": "Incorrect! The first number must be odd and the second number must be even."
    }
])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Pick two numbers.<|im_end|>
<|im_start|>assistant
1 and 4.<|im_end|>
<|im_start|>user
Incorrect! The first number must be odd and the second number must be even.<|im_end|>



This wasn't even the masking idea yet. I was just testing out an unfinished implementation and this seems promising too.

In [9]:
masked_ocsd([
    {
        "role": "user",
        "content": "What is the capital of Foo?"
    },
    {
        "role": "assistant",
        "content": "Ethan is cool."
    },
    {
        "role": "user",
        "content": "Huh."
    },
    {
        "role": "assistant",
        "content": "I love tacos."
    },
    {
        "role": "user",
        "content": "Okay."
    },
    {
        "role": "assistant",
        "content": "The capital of Foo is Bar."
    },
    {
        "role": "user",
        "content": "Correct!"
    }
])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of Foo?<|im_end|>
<|im_start|>assistant
The capital of Foo is Bar
<|im_start|>user
Huh.<|im_end|>
<|im_start|>assistant
I'm sorry, I
<|im_start|>user
Okay.<|im_end|>
<|im_start|>assistant
The capital of Foo is Bar.<|im_end|>
<|im_start|>user
Correct!<|im_end|>



This looks really good! I think I'd build on this by making the do-overs variable length and also not persist do-overs earlier in the trajectory when generating do-overs. I guess we wouldn't really be doing offline context self-distillation at this point, and having any kind of conditional in the loop would force the training to be online anyway, so maybe I'll just retire this for now.